# train_colab.ipynb — SPEC v2.1 mục 10

**Chỉ chứa lệnh chạy — không có logic.** Mọi logic nằm trong `src/`, test được qua CI (`tests/unit`), không nằm ở đây.

Thứ tự giả định: bạn đã chạy `dvc repro` (data pipeline — `augment_masks` + `make_splits`) và `dvc push` **trên laptop** trước (MaskTheFace/dlib là việc CPU, không cần GPU — xem hướng dẫn chạy riêng). Notebook này chỉ `dvc pull` lại kết quả đó + weight pretrained, rồi train/eval trên GPU Colab.

Trước khi chạy: điền `CHECKPOINT_DIR` ở Cell 2 (đường dẫn trên Google Drive của bạn) và đảm bảo đã set 3 Colab Secrets: `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` (menu 🔑 bên trái Colab — xem hướng dẫn chạy riêng mục 1.4).

## 1. Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content
!git clone <repo-url> face-tracking-mlops
%cd face-tracking-mlops
# Checkout đúng commit/tag đã dùng để chạy dvc repro trên laptop, nếu khác main:
# !git checkout <commit-hoặc-tag>

## 2. Cấu hình đường dẫn checkpoint (Drive)

In [ ]:
import os

# SỬA đường dẫn này theo Drive của bạn — giữ NGUYÊN giữa các lần chạy để
# resume hoạt động đúng (CheckpointManager tìm checkpoint theo đúng thư mục này).
os.environ["CHECKPOINT_DIR"] = "/content/drive/MyDrive/face-tracking-mlops/checkpoints"
!mkdir -p "$CHECKPOINT_DIR"

## 3. Cài đặt (`pip install -e .` + `requirements/train.txt`)

In [ ]:
%pip install -e .
%pip install -r requirements/train.txt
%pip install "dvc[s3]==3.67.1"

## 4. MLflow credentials (Colab Secrets — menu 🔑 bên trái, KHÔNG hardcode token vào notebook)

In [ ]:
from google.colab import userdata

os.environ["MLFLOW_TRACKING_URI"] = userdata.get("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("MLFLOW_TRACKING_PASSWORD")
# DagsHub token dùng chung cho `dvc pull`/`dvc push` (Cell 5, 8) — DVC đọc qua
# remote đã cấu hình sẵn trong .dvc/config (committed), chỉ cần credential:
os.environ["DAGSHUB_TOKEN"] = userdata.get("DAGSHUB_TOKEN")
# Xóa credential local cũ nếu notebook đã chạy trước đó với remote sai schema.
from pathlib import Path
Path(".dvc/config.local").unlink(missing_ok=True)
!dvc remote modify dagshub-storage url s3://dvc
!dvc remote modify dagshub-storage endpointurl https://dagshub.com/ltnguyen53/face.s3
!dvc remote modify dagshub-storage --local access_key_id "$DAGSHUB_TOKEN"
!dvc remote modify dagshub-storage --local secret_access_key "$DAGSHUB_TOKEN"

## 5. `dvc pull` — data đã xử lý (từ `dvc repro` trên laptop) + weight pretrained

In [ ]:
!dvc pull

## 6. Train

Ngắt giữa chừng (hết giờ Colab, rớt mạng)? Chạy LẠI đúng cell này — `CheckpointManager` tự tìm checkpoint khớp `config_hash` trong `CHECKPOINT_DIR` và resume đúng epoch/step (đã verify bằng test tích hợp thật, xem `tests/unit/test_train.py`), không train lại từ đầu.

In [ ]:
!python -m src.training.train \
    --config configs/finetune_occlusion.yaml \
    --export-config configs/export.yaml \
    --resume-dir "$CHECKPOINT_DIR"

## 7. Eval — threshold sweep + report + đăng ký model (nếu tốt hơn Production hiện tại)

Ghi `reports/eval_metrics.json`, cập nhật `configs/inference.yaml` (threshold đã khoá), log MLflow, và tự đăng ký model version mới vào Registry nếu `test_overall_tar` tốt hơn bản Production hiện tại — **không tự promote Production**, bạn tự xem lại trên DagsHub UI rồi chuyển Staging → Production.

In [ ]:
!python -m src.evaluation.run_eval \
    --config configs/finetune_occlusion.yaml \
    --inference-config configs/inference.yaml \
    --export-config configs/export.yaml \
    --checkpoint-dir "$CHECKPOINT_DIR"

In [ ]:
!cat reports/eval_metrics.json

## 8. Push kết quả lên DagsHub

`dvc push` đẩy artifact (nếu `dvc.yaml` có stage track checkpoint/report — hiện tại training/eval KHÔNG phải DVC stage, xem hướng dẫn chạy riêng, nên bước này chủ yếu để phòng trường hợp bạn tự `dvc add` thêm). `configs/inference.yaml` và `reports/eval_metrics.json` là file text thường — commit qua git bình thường, không qua DVC.

In [ ]:
!dvc push
!git add configs/inference.yaml reports/eval_metrics.json
!git commit -m "eval: cập nhật threshold + report sau lần train này"
!git push